In [ ]:
import pandas as pd
import numpy as np

from vpop_calibration import *

%load_ext autoreload
%autoreload 2

In [ ]:
df = pd.read_csv("Mavoglurant_Dataset.csv")


obs_df = (
    df.loc[df["EVID"] == 0]
    .rename(columns={"ID": "id", "TIME": "time", "DV": "value", "DOSE": "Dose"})[
        ["id", "time", "value", "Dose", "WT"]
    ]
    .astype({"value": "float"})
)
obs_df["time"] = obs_df["time"].apply(lambda t: t * 60 * 60)
obs_df["value"] = obs_df["value"].apply(lambda t: np.log(t))
obs_df["output_name"] = "logC15"
obs_df["protocol_arm"] = "identity"
display(df.head())
display(obs_df.head())

In [ ]:
struct_model = StructuralSbml(
    model_path="CM_Mavoglurant.xml",
    inputs=["KbBR", "CLint", "KbMU", "KbAD", "KbBO", "KbRB", "Dose", "WT"],
    outputs=["logC15"],
)

struct_model.rr.setIntegrator("cvode")
integ = struct_model.rr.integrator
integ.setValue("stiff", True)
integ.setValue("relative_tolerance", 1e-6)
integ.setValue("absolute_tolerance", 1e-6)
integ.setValue("initial_time_step", 1e-6)

In [ ]:
prior_pdu = {
    "pdu": {
        "CLint": {"prior": np.exp(7.6), "prior_omega": 4},
        "KbBR": {"prior": np.exp(1.1), "prior_omega": 0.5},
        "KbMU": {"prior": np.exp(0.3), "prior_omega": 0.5},
        "KbBO": {"prior": np.exp(0.03), "prior_omega": 0.5},
        "KbAD": {"prior": np.exp(2), "prior_omega": 0.5},
        "KbRB": {"prior": np.exp(0.3), "prior_omega": 0.5},
    },
    "pdk": {"WT", "Dose"},
    "error_model": {
        "logC15": {"error_type": "additive", "sigma": 0.5},
    },
}
config = Config(
    saem=SaemConfigDict(
        nb_iter_burnin=0,
        nb_iter_learning=100,
        nb_iter_smoothing=100,
        plot_frames=5,
    ),
    nlme=NlmeConfigDict(nb_chains=1),
)
nlme_model = NlmeModel(
    df=obs_df, prior_params=prior_pdu, structural_model=struct_model, config=config
)

In [ ]:
nlme_model.optimizer.run()

In [ ]:
nlme_model.diagnostics.sample_conditional_distribution(nb_samples=100)

In [ ]:
nlme_model.plot.map_estimates()

In [ ]:
nlme_model.plot.map_estimates_gof()

In [ ]:
nlme_model.plot.weighted_residuals("iwres")

In [ ]:
nlme_model.plot.vpc()

In [ ]:
nlme_model.plot.conditional_codistributions()